# JAX Extension · Fourier Neural Operator for Burgers — Lab 1 in JAX

<img src="../../assets/logos/amiad.png" width="120"/> <img src="../../assets/logos/cea.png" width="120"/> <img src="../../assets/logos/lmps.png" width="120"/> <img src="../../assets/logos/borelli.png" width="120"/>

**Authors:** E. Noëlé, Gatti, Lehmann, Millet, Clouteau<br>
**Affiliations:** AMIAD, CEA, LMPS, Borelli<br>
**Contact:** elodie.noele@TODO.example

**LETMA 2026 Summer School · optional extension** — the same lab as Lab 1
(`lab_fno_student.ipynb`), reimplemented in **JAX + Equinox + Optax** instead of PyTorch.
The data, the hypotheses (H1–H7 of Lab 1 §1), the architecture and every experiment are
identical (the data ships in `data/burgers_jax.npz`, generated once with the very same seeds —
so this notebook needs **neither torch nor the tutorial packages**, only JAX), and you should get
comparable but not bit-identical numbers (different random streams, different kernels). What changes is the *programming model*:

> **PyTorch**: object-oriented and mutable — a module *holds* parameters, `loss.backward()`
> writes gradients into them, the optimizer mutates them in place.
> **JAX**: functional and immutable — a model is a *pytree* (nested dict/tuple of arrays),
> gradients are pure functions of that pytree (`jax.grad`), and "updating the parameters"
> produces a *new* pytree (`optax.apply_updates`). Equinox gives modules-without-mutation;
> Optax is the optimizer library. Batching is `jax.vmap` (no DataLoader), and `jax.jit`
> compiles each step once — then it flies.

If you have done Lab 1, jump straight to §2 (the spectral conv exercise, in JAX) and §4
(the loss landscape and the optimizer race — the JAX version is a nice showcase of
functional style). Everything runs on CPU; a GPU only speeds the corner plot up.


## §0 · Setup — run me first

In [ ]:
# Portable setup: runs unchanged on the Hedy cluster, a laptop, Binder or Google Colab. Run this cell first.
import os, sys, math, time
from pathlib import Path
if 'google.colab' in sys.modules:                      # Google Colab: fetch the repository once
    import importlib.util, subprocess
    COLAB_ROOT = Path('/content/Tutorial_FNO_GNN_LETMA_AutumnSchool_2026')
    if not (COLAB_ROOT / 'fno_tuto').is_dir():
        subprocess.run(['git', 'clone', '-q', '--depth', '1',
                        'https://github.com/noeleel/Tutorial_FNO_GNN_LETMA_AutumnSchool_2026.git',
                        str(COLAB_ROOT)], check=True)
    missing = [p for p in ('equinox', 'optax') if importlib.util.find_spec(p) is None]
    if missing:                                        # the only deps Colab does not preinstall
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
    os.environ['HEDY_PROJECT_ROOT'] = str(COLAB_ROOT)
project_root = next((p for p in [Path(os.environ.get('HEDY_PROJECT_ROOT', Path.cwd())),
                                  Path.cwd(), *Path.cwd().parents]
                     if (p / 'data' / 'burgers_jax.npz').is_file()), None)
if project_root is None:
    raise RuntimeError('data/burgers_jax.npz not found: open this notebook inside the '
                       'tutorial project directory (and run scripts/make_burgers_npz.py once).')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from IPython import get_ipython
PLOT_MODE = os.environ.get('LAB_PLOT_MODE', 'inline').strip().lower()
if PLOT_MODE == 'tk':                      # tk needs a WORKING X display; test it for real
    try:
        import tkinter
        tkinter.Tk().destroy()
    except Exception:
        print('no usable X display -> falling back to inline plots (ssh -Y for tk windows)')
        PLOT_MODE = 'inline'
get_ipython().run_line_magic('matplotlib', PLOT_MODE)

import numpy as np
import jax
import jax.numpy as jnp
import equinox as eqx
import optax
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11, 'axes.titlesize': 12,
                     'axes.labelsize': 11, 'legend.fontsize': 10})
HEDY_SESSION = None
try:
    from hedy_euporie.profiler import begin_session
    HEDY_SESSION = begin_session('jax_fno_student', backend='jax')
except Exception:
    print('Hedy session profiling not available here (local/laptop run): continuing without it.')

def show_anim(fig, func, frames, interval=200, blit=False):
    """Inline JS animation in Jupyter/JupyterLab. On Hedy (Euporie cannot play the
    JavaScript animation) a static filmstrip of a few key frames is shown instead."""
    n_frames = frames if isinstance(frames, int) else len(frames)
    if HEDY_SESSION is not None:              # on Hedy: static filmstrip, no JS player
        picks = np.unique(np.linspace(0, max(n_frames - 1, 0), 6).round().astype(int))
        snaps = []
        for f in picks:
            func(int(f))
            fig.canvas.draw()
            snaps.append(np.asarray(fig.canvas.buffer_rgba())[:, :, :3])
        plt.close(fig)
        fig2, ax2 = plt.subplots(1, len(picks), figsize=(2.4 * len(picks), 2.9))
        for a, s, f in zip(np.atleast_1d(ax2), snaps, picks):
            a.imshow(s); a.set_xticks([]); a.set_yticks([])
            a.set_title(f'frame {int(f) + 1}/{n_frames}', fontsize=8)
        fig2.tight_layout(); plt.show()
    else:
        ani = animation.FuncAnimation(fig, func, frames=frames, interval=interval, blit=blit)
        plt.close(fig)          # suppress the empty static frame
        display(HTML(ani.to_jshtml()))

print(f'Python {sys.version.split()[0]} | JAX {jax.__version__} | devices {jax.devices()}')
print(f'backend: {jax.default_backend()} — the lab runs on CPU too, just slower')


## §1 · The data — same Burgers problem, same hypotheses as Lab 1

Same equation $\partial_t u + u\,\partial_x u = \nu\,\partial_x^2 u$, same Gaussian random
field inputs (Matérn spectrum, $\alpha = 2.5$, $\tau = 12$), same target $u(\cdot, T)$ with
$T = 0.3$, $\nu = 0.008$ — and the same hypotheses **H1–H7** (well-posed operator, i.i.d.
sampling, split by index, normalization from training statistics, resolution invariance,
spectral decay, relative-$L^2$ metric). Lab 1 §1 states and justifies each of them; nothing
changes because we switched framework, so we do not restate them here. We still *look* at the
data, because that habit transfers to your own problems.

In [ ]:
DATA = np.load(project_root / 'data' / 'burgers_jax.npz')   # the same samples as Lab 1, torch-free
N_RES = 64
n_train, n_val, n_test = 128, 32, 32
Xtr_np, Ytr_np = DATA['X64'][:n_train], DATA['Y64'][:n_train]
Xva_np, Yva_np = DATA['X64'][n_train:-n_test], DATA['Y64'][n_train:-n_test]
Xte_np, Yte_np = DATA['X64'][-n_test:], DATA['Y64'][-n_test:]

# normalization (H5): statistics from the training inputs/outputs only, then frozen
x_mu, x_sd = Xtr_np.mean(), Xtr_np.std()
y_mu, y_sd = Ytr_np.mean(), Ytr_np.std()
def enc(t): return ((t - x_mu) / x_sd).astype(jnp.float32)          # input encoder
def enc_y(t): return ((t - y_mu) / y_sd).astype(jnp.float32)        # target encoder
def dec_y(t): return t * y_sd + y_mu                                # back to physical units

Xtr_jnp, Ytr_jnp = enc(Xtr_np), enc_y(Ytr_np)     # (B, 1, N) — JAX arrays, ready for vmap
x_grid = np.linspace(0, 1, N_RES, endpoint=False)
print('splits (B, C=1, N):', Xtr_np.shape, Xva_np.shape, Xte_np.shape)

fig, ax = plt.subplots(1, 3, figsize=(12.5, 3.2))
ax[0].hist(Xtr_np.ravel(), bins=60, color='C0', alpha=0.8, label='u0 (input)')
ax[0].hist(Ytr_np.ravel(), bins=60, color='C3', alpha=0.55, label='u(.,T) (output)')
ax[0].set_xlabel('field value'); ax[0].legend(fontsize=8)
ax[0].set_title('INPUT and OUTPUT value distributions')
nrm_in = np.linalg.norm(Xtr_np.reshape(n_train, -1), axis=1)
nrm_out = np.linalg.norm(Ytr_np.reshape(n_train, -1), axis=1)
ax[1].hist(nrm_in, bins=24, color='C0', alpha=0.8, label=r'$||u_0||_2$')
ax[1].hist(nrm_out, bins=24, color='C3', alpha=0.7, label=r'$||u_T||_2$')
ax[1].set_xlabel('per-sample L2 norm'); ax[1].legend(fontsize=8)
ax[1].set_title('per-sample amplitudes (the imbalance of section 4)')
ax[2].hist(nrm_out / nrm_in, bins=24, color='C2', alpha=0.85)
ax[2].set_xlabel(r'$||u(\cdot,T)||\,/\,||u_0||$')
ax[2].set_title('how much the operator damps each sample')
plt.tight_layout(); plt.show()
print(f'amplitude spread: ||u0|| varies by a factor {nrm_in.max()/nrm_in.min():.1f}; '
      f'the operator maps every input to ~{(nrm_out/nrm_in).mean():.0%} of its norm')

## §2 · The spectral convolution, in JAX — Exercise F1

Exactly Lab 1 §2: the operator's global action is learned as a complex matrix per Fourier
mode, $\hat v^{[k]} \mapsto R^{[k]} \hat v^{[k]}$ for $k < m$, zero beyond. `jnp.fft.rfft`
mirrors `torch.fft.rfft` mode for mode; the exercise is to wire the four steps together.

In [ ]:
# ---- Exercise F1 · 5 minutes: implement the spectral convolution in JAX ----
# Contract:  x (C_in, N) ONE sample, w (C_in, C_out, modes) complex  ->  (C_out, N) real.
#   (batching is done by jax.vmap later: every layer sees ONE sample)
# Steps (identical to Lab 1, but with jnp):  1) rfft along the LAST axis, norm='ortho'
#   2) keep only the first m = min(modes, ...) coefficients
#   3) multiply mode-by-mode: out^[k] = w[:, :, k] @ xh^[k]  (an einsum 'ik,iok->ok')
#   4) irfft with n=N (mandatory for odd grid sizes) back to physical space
# Not passing? The checker below falls back to the reference and the lab continues.
def my_spectral_conv(x, w, modes):
    n = x.shape[-1]
    # TODO 1: real FFT of x along the last axis (norm='ortho')
    # xh = ...
    # TODO 2: truncate to the first m = min(modes, ...) coefficients
    # TODO 3: einsum 'ik,iok->ok' between xh[..., :m] and w[..., :m]
    # TODO 4: inverse real FFT with n=n (norm='ortho')
    # return ...
    raise NotImplementedError('your turn: 4 lines, see the comments above')

In [ ]:
# ---- check F1 against the reference (falls back silently if not passing) ----
def ref_spectral_conv(x, w, modes):
    n = x.shape[-1]
    xh = jnp.fft.rfft(x, axis=-1, norm='ortho')
    m = min(modes, xh.shape[-1])
    out = jnp.einsum('ik,iok->ok', xh[..., :m], w[..., :m])
    return jnp.fft.irfft(out, n=n, norm='ortho')

passed_f1 = False
try:
    kb = jax.random.PRNGKey(42)
    xt = jax.random.normal(kb, (3, 64))
    wt = 0.1j * jax.random.normal(jax.random.PRNGKey(43), (3, 5, 16)) + \
         0.1 * jax.random.normal(jax.random.PRNGKey(44), (3, 5, 16))
    err = float(jnp.abs(my_spectral_conv(xt, wt, 16) - ref_spectral_conv(xt, wt, 16)).max())
    passed_f1 = err < 1e-5
    print('F1 check:', f'PASS (max diff {err:.2e})' if passed_f1 else f'FAIL (max diff {err:.2e})')
except NotImplementedError:
    print('F1 check: not implemented yet')
except Exception as e:
    print('F1 check: error ->', type(e).__name__, e)
spectral_conv = my_spectral_conv if passed_f1 else ref_spectral_conv
print('active spectral convolution:', 'YOURS' if passed_f1 else 'reference fallback')

In [ ]:
# ---- the mode-truncation demo: H7 says the energy lives in the first modes ----
u = Ytr_np[0, 0]
coef = np.fft.rfft(u, norm='ortho')
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(x_grid, u, color='C0', lw=2, label='truth')
for m in [1, 2, 4, 8, 16, 32]:
    c = np.zeros_like(coef); c[:m] = coef[:m]
    r = np.fft.irfft(c, n=N_RES, norm='ortho')
    err = np.linalg.norm(r - u) / np.linalg.norm(u)
    ax[0].plot(x_grid, r, ls='--', lw=1, label=f'm={m} ({err:.3f})')
ax[0].legend(fontsize=7); ax[0].set_xlabel('x'); ax[0].set_ylabel('u')
ax[0].set_title('keeping m Fourier modes (same experiment as Lab 1)')
ax[1].semilogy(np.abs(coef) + 1e-12, color='C0')
ax[1].set_xlabel('mode k'); ax[1].set_ylabel(r'$|\hat u_k|$')
ax[1].set_title('H7: spectral decay of a target field'); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## §3 · The full FNO1d in Equinox, and its training with Optax

Same architecture as Lab 1 §3 — $a(x) \xrightarrow{P} v_0 \xrightarrow{\text{block}} \cdots
\xrightarrow{\text{block}} v_L \xrightarrow{Q} u(x)$ with $v_{\ell+1} = \mathrm{GELU}(K_\ell v_\ell + W_\ell v_\ell)$ —
and the same reading of the latent space: **$P$ embeds the physical field into $w = 32$ learned
channels per grid point; all the spectral mixing and all the nonlinearity happen in that learned
representation; $Q$ reads it back to physical units.** The width is the rank of the
representation; nothing in the architecture fixes what the channels mean.

JAX-specific translation of the hyperparameter table of Lab 1 §3:

| hyperparameter | value | where it lives here |
|---|---|---|
| `width` $w$ | 32 | `FNO1dJAX(..., width=32)` — also the width of $P$ |
| `modes` $m$ | 16 | above the 99.9%-energy mark of §2 (H7) |
| `n_layers` $L$ | 4 | four blocks, each `GELU(spectral + pointwise)` |
| `proj_hidden` | 128 | width of $Q$'s hidden layer |
| loss | relative $L^2$ | `rel_l2` below (the FNO literature's metric) |
| optimizer | AdamW, lr $2\!\cdot\!10^{-3}$ + cosine decay | `optax.adamw` + `cosine_decay_schedule` |
| `weight_decay` | $10^{-4}$ | via an optax **mask**: decay on the real pointwise weights only, never on the complex spectral weights |
| epochs / batch | 60 / 32 | batching by `jax.vmap` + index slicing (no DataLoader) |

**Exercise F2** is the forward pass — in JAX it is a *pure function* of the model, which is
precisely the point of the exercise.

> **Why the spectral weights are stored as real (Re, Im) pairs.** The math is complex — each
> retained mode $k$ multiplies $\hat v^{[k]}$ by a complex matrix $R^{[k]}$ — but we store
> those weights as two *real* arrays and assemble them (`re + 1j*im`) at call time. The reason
> is a genuine JAX pitfall: `jax.grad` of a real-valued loss with respect to **complex**
> parameters follows the conjugate (Wirtinger) convention, and feeding those complex gradients
> naively to Optax *stalls* training on this problem (we measured it: rel-L2 0.36 after 60
> epochs with complex storage, 0.09 with the real storage). PyTorch defines complex gradients
> as $\partial L/\partial \mathrm{Re} + i\,\partial L/\partial \mathrm{Im}$ and trains fine
> with complex tensors — this is one of the few places where the two frameworks genuinely
> differ. Real storage is the standard JAX fix: same function space, well-behaved gradients,
> and Lab 1's rule "never weight-decay the complex weights" simply becomes "don't decay the
> 3D Re/Im arrays" in the Optax mask below.

In [ ]:
# ---- the building blocks: Equinox modules = pytrees with methods, nothing more ----
class Pointwise(eqx.Module):
    weight: jax.Array                     # (C_in, C_out) — a 1x1 convolution without bias
    def __init__(self, key, c_in, c_out):
        lim = 1.0 / math.sqrt(c_in)
        self.weight = lim * (2 * jax.random.uniform(key, (c_in, c_out)) - 1)
    def __call__(self, x):                # (C_in, N) -> (C_out, N), one sample at a time
        return jnp.einsum('ik,io->ok', x, self.weight)

class SpectralBlock(eqx.Module):
    w_re: jax.Array                       # (w, w, modes) real part of the complex weights
    w_im: jax.Array                       # (w, w, modes) imaginary part
    w_pw: jax.Array                       # (w, w) real
    def __init__(self, key, width, modes):
        k1, k1b, k2 = jax.random.split(key, 3)
        scale = 1.0 / (width * width)     # same small init as Lab 1: start near zero
        self.w_re = scale * jax.random.normal(k1, (width, width, modes))
        self.w_im = scale * jax.random.normal(k1b, (width, width, modes))
        lim = 1.0 / math.sqrt(width)
        self.w_pw = lim * (2 * jax.random.uniform(k2, (width, width)) - 1)
    def spectral(self, x):                # (w, N) -> (w, N), uses YOUR F1
        w = self.w_re + 1j * self.w_im    # the complex weights, ASSEMBLED from real parts
        return spectral_conv(x, w, self.w_im.shape[-1])
    def pointwise(self, x):
        return jnp.einsum('ik,io->ok', x, self.w_pw)

class FNO1dJAX(eqx.Module):
    lift: Pointwise                       # P: (2 = u + grid) -> width
    blocks: list
    proj1: Pointwise                      # Q: width -> proj_hidden -> 1
    proj2: Pointwise
    def __init__(self, key, in_ch=1, width=32, modes=16, n_layers=4, proj_hidden=128):
        keys = jax.random.split(key, n_layers + 3)
        self.lift = Pointwise(keys[0], in_ch + 1, width)
        self.blocks = [SpectralBlock(keys[1 + i], width, modes) for i in range(n_layers)]
        self.proj1 = Pointwise(keys[-2], width, proj_hidden)
        self.proj2 = Pointwise(keys[-1], proj_hidden, 1)

key = jax.random.PRNGKey(0)
fno = FNO1dJAX(key)
params_all = eqx.filter(fno, eqx.is_array)
n_par = sum(jnp.size(l) for l in jax.tree_util.tree_leaves(params_all))
print(f'FNO1dJAX(width=32, modes=16, 4 blocks, proj_hidden=128): {int(n_par):,} real parameters '
      '(the complex spectral weights are stored as real (Re, Im) pairs: see the note above)')

In [ ]:
# ---- Exercise F2 · 5 minutes: assemble the forward pass (functional style) ----
# Contract: model (FNO1dJAX, built below), x (C_in=1, N) ONE sample -> (C_out=1, N).
# The pipeline is exactly Lab 1's:  append the coordinate channel,  then P (model.lift),
#   then L blocks  v <- GELU(spectral(v) + pointwise(v)),  then Q (model.proj1, GELU, proj2).
# In JAX the model is DATA: your function receives it and READS its submodules — no mutation.
# Not passing? The checker falls back to the reference forward.
def my_fno_forward(model, x):
    n = x.shape[-1]
    # TODO 1: grid = jnp.linspace(0, 1, n, endpoint=False) reshaped to (1, N)
    # TODO 2: v = model.lift(jnp.concatenate([x, grid], axis=0))
    # TODO 3: for blk in model.blocks: v = jax.nn.gelu(blk.spectral(v) + blk.pointwise(v))
    # TODO 4: return model.proj2(jax.nn.gelu(model.proj1(v)))
    raise NotImplementedError('your turn: 4 lines, see the comments above')

In [ ]:
# ---- check F2 against the reference forward (fallback keeps the lab on track) ----
def ref_fno_forward(model, x):
    n = x.shape[-1]
    grid = jnp.linspace(0.0, 1.0, n, endpoint=False)[None]
    v = model.lift(jnp.concatenate([x, grid], axis=0))
    for blk in model.blocks:
        v = jax.nn.gelu(blk.spectral(v) + blk.pointwise(v))
    return model.proj2(jax.nn.gelu(model.proj1(v)))

passed_f2 = False
try:
    xt = Xtr_jnp[:2]
    err = float(jnp.abs(jax.vmap(my_fno_forward, in_axes=(None, 0))(fno, xt)
                        - jax.vmap(ref_fno_forward, in_axes=(None, 0))(fno, xt)).max())
    passed_f2 = err < 1e-5
    print('F2 check:', f'PASS (max diff {err:.2e})' if passed_f2 else f'FAIL (max diff {err:.2e})')
except NotImplementedError:
    print('F2 check: not implemented yet')
except Exception as e:
    print('F2 check: error ->', type(e).__name__, e)
fno_forward = my_fno_forward if passed_f2 else ref_fno_forward
print('active forward pass:', 'YOURS' if passed_f2 else 'reference fallback')

In [ ]:
# ---- training with Optax: AdamW (masked weight decay) + cosine schedule ----
def rel_l2(pred, y):
    d = jnp.linalg.norm((pred - y).reshape(len(y), -1), axis=-1)
    n = jnp.linalg.norm(y.reshape(len(y), -1), axis=-1)
    return jnp.mean(d / n)

def batch_forward(model, xb):             # vmap replaces the DataLoader+forward
    return jax.vmap(fno_forward, in_axes=(None, 0))(model, xb)

def make_optimizer(lr=2e-3, wd=1e-4, total_steps=240):
    def decay_mask(p):        # decay ONLY the real 2D pointwise weights, exactly as Lab 1
        return jax.tree_util.tree_map(   # (the spectral Re/Im arrays are 3D: no decay)
            lambda x: bool(x.ndim == 2), p)
    sched = optax.cosine_decay_schedule(lr, total_steps)
    return optax.adamw(learning_rate=sched, weight_decay=wd, mask=decay_mask)

def fit(model, epochs=60, batch=32, lr=2e-3, wd=1e-4, key_train=jax.random.PRNGKey(1),
        snap_every=None, n_snap=3):
    """The Lab-1 training loop, translated: vmap batches, optax steps, snapshots."""
    params, static_ = eqx.partition(model, eqx.is_array)   # the model IS (params, static)

    @eqx.filter_jit
    def train_step(params, opt_state, xb, yb):
        def loss_of(p):
            return rel_l2(batch_forward(eqx.combine(p, static_), xb), yb)
        loss, grads = jax.value_and_grad(loss_of)(params)
        updates, opt_state = optimizer.update(grads, opt_state, params=params)
        return optax.apply_updates(params, updates), opt_state, loss

    steps = math.ceil(n_train / batch) * epochs
    optimizer = make_optimizer(lr, wd, steps)
    opt_state = optimizer.init(params)
    Xva_j = enc(Xva_np); Yva_j = enc_y(Yva_np)
    hist = {'train': [], 'val': []}; snaps = []
    for ep in range(1, epochs + 1):
        key_train, sub = jax.random.split(key_train)
        perm = jax.random.permutation(sub, n_train)
        tot, nb = 0.0, 0
        for i in range(0, n_train, batch):
            idx = perm[i:i + batch]
            params, opt_state, loss = train_step(params, opt_state,
                                                 Xtr_jnp[idx], Ytr_jnp[idx])
            tot += float(loss); nb += 1
        model = eqx.combine(params, static_)
        pv = rel_l2(batch_forward(model, Xva_j), Yva_j)
        hist['train'].append(tot / nb); hist['val'].append(float(pv))
        if snap_every and (ep % snap_every == 0 or ep == 1):
            snaps.append((ep, np.array(dec_y(batch_forward(model, Xva_j[:n_snap])))))
    return model, hist, snaps

fno, hist, snaps = fit(fno, epochs=60, snap_every=5)
print(f"final validation rel-L2: {hist['val'][-1]:.4f}  "
      f"(Lab 1 / PyTorch reached ~0.076 — JAX should land in the same range)")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(hist['train'], label='training loss (what the optimizer sees)')
ax.plot(hist['val'], label='validation rel-L2 (what we report)')
ax.set_xlabel('epoch'); ax.set_ylabel('loss / error'); ax.legend(); ax.grid(alpha=0.3)
ax.set_title('Same curves as Lab 1: both fall together, no overfitting at this size')
plt.show()

In [ ]:
# ---- animation: the operator being learned, epoch by epoch (JAX edition) ----
fig, ax = plt.subplots(1, 4, figsize=(12.5, 3.1))
eps = [s[0] for s in snaps]
curves = []
for j in range(3):
    ax[j].plot(x_grid, Yva_np[j, 0], color='C0', lw=2, label='truth')
    (ln,) = ax[j].plot(x_grid, snaps[0][1][j, 0], color='C3', ls='--', lw=1.8, label='JAX FNO')
    ax[j].set_ylim(-3.2, 3.2); ax[j].grid(alpha=0.3); ax[j].set_xlabel('x')
    curves.append(ln)
ax[0].set_ylabel('u'); ax[0].legend()
(loss_ln,) = ax[3].plot(eps, [hist['val'][e - 1] for e in eps], color='0.4')
(dot,) = ax[3].plot([], [], 'o', color='C3')
ax[3].set_xlabel('epoch'); ax[3].set_ylabel('val rel-L2'); ax[3].grid(alpha=0.3)
title = fig.suptitle('', y=1.02)

def update(f):
    ep, preds = snaps[f]
    for j in range(3):
        curves[j].set_ydata(preds[j, 0])
    dot.set_data([ep], [hist['val'][ep - 1]])
    title.set_text(f'epoch {ep}: predictions vs truth on 3 unseen validation fields')

show_anim(fig, update, frames=len(snaps), interval=420)

## §4 · The loss landscape and the optimizer race — in JAX

Exactly Lab 1 §4.2: pick the two directions $d_1$ (init → trained) and $d_2$ (random) around
the trained solution $\theta^\star$, plot the true loss on the plane
$\theta^\star + \alpha d_1 + \beta d_2$, and re-run four optimizers **from the same
initialization, each until it actually reaches the trained loss $L^\star$** (or exhausts its
budget).

> **What is rigorous here** (same caveats as Lab 1): every landscape point is a true loss of
> a true parameter vector — no approximation; the trajectory projections are exact. The only
> approximation is reading a 2-D slice as representative of a ~140k-dimensional geometry.
>
> **The JAX twist:** moving on the plane is `jax.tree_util.tree_map` over the *parameter pytree*
> — the same operation at any depth, because in JAX the model IS the pytree. Interpolating a
> torch model would require flattening every tensor by hand.

In [ ]:
# ---- build the landscape on the plane spanned by (d1, d2) around theta* ----
import jax.tree_util as jtu
params_star, static = eqx.partition(fno, eqx.is_array)
params_init, _ = eqx.partition(FNO1dJAX(jax.random.PRNGKey(0)), eqx.is_array)

def t_sub(a, b):   return jtu.tree_map(lambda x, y: x - y, a, b)
def t_scale(a, s): return jtu.tree_map(lambda x: x * s, a)
def t_dot(a, b):
    return sum(jnp.real(jnp.vdot(jnp.ravel(x), jnp.ravel(y)))
               for x, y in zip(jtu.tree_leaves(a), jtu.tree_leaves(b)))
def t_norm(a): return float(jnp.sqrt(t_dot(a, a)))

d1 = t_sub(params_init, params_star); s0 = t_norm(d1); d1 = t_scale(d1, 1.0 / s0)
leaves, treedef = jtu.tree_flatten(params_star)
kr = jax.random.split(jax.random.PRNGKey(1), len(leaves))
d2 = treedef.unflatten([jax.random.normal(k, l.shape, l.dtype) for k, l in zip(kr, leaves)])
d2 = t_scale(d2, 1.0 / t_norm(d2))
print(f'||theta_init - theta*|| = {s0:.3f}  (the landscape spans this scale)')

xb, yb = Xtr_jnp[:32], Ytr_jnp[:32]      # one fixed batch: fast and deterministic

@jax.jit
def plane_loss(a, b):
    p = jtu.tree_map(lambda s_, u, v: s_ + a * u + b * v, params_star, d1, d2)
    m = eqx.combine(p, static)
    return rel_l2(batch_forward(m, xb), yb)

G = 21
alphas = np.linspace(-1.15 * s0, 1.15 * s0, G)
betas = np.linspace(-1.15 * s0, 1.15 * s0, G)
# 0-d jnp arrays (NOT Python floats): one single jit compilation for all 21x21 points
Z = np.array([[float(plane_loss(jnp.float32(a), jnp.float32(b))) for b in betas]
              for a in alphas])
L_star = float(plane_loss(jnp.float32(0.0), jnp.float32(0.0)))
print(f'landscape: {G}x{G} = {G*G} true-loss evaluations | trained loss L* = {L_star:.4f}')

In [ ]:
# ---- the race: four optax optimizers, same start, each chasing L* ----
def trajectory(opt, max_steps=700, tol=1.05):
    params = params_init
    opt_state = opt.init(params)
    path, losses = [], []

    @jax.jit
    def step(params, opt_state):
        m = eqx.combine(params, static)
        loss, grads = jax.value_and_grad(
            lambda p: rel_l2(batch_forward(eqx.combine(p, static), xb), yb))(params)
        updates, opt_state = opt.update(grads, opt_state, params=params)
        return optax.apply_updates(params, updates), opt_state, loss

    for t in range(max_steps):
        params, opt_state, loss = step(params, opt_state)
        path.append((float(t_dot(t_sub(params, params_star), d1)),
                    float(t_dot(t_sub(params, params_star), d2))))
        losses.append(float(loss))
        if losses[-1] <= tol * L_star:
            break
    return np.array(path, dtype=float), np.array(losses)

runs = {'SGD lr 1e-2':       trajectory(optax.sgd(1e-2)),
        'SGD lr 5e-2':       trajectory(optax.sgd(5e-2)),
        'SGD+momentum 5e-2': trajectory(optax.sgd(5e-2, momentum=0.9)),
        'Adam lr 2e-3':      trajectory(optax.adam(2e-3))}

print(f'{"optimizer":20s} {"steps":>6s} {"final loss":>11s}   verdict')
for name, (path, ls) in runs.items():
    arrived = ls[-1] <= 1.05 * L_star
    print(f'{name:20s} {len(ls):6d} {ls[-1]:11.4f}   '
          + ('ARRIVED at the trained loss' if arrived else 'stopped at max steps, still far above L*'))

from matplotlib.colors import LogNorm
fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.8))
ext = [alphas[0], alphas[-1], betas[0], betas[-1]]
im = ax[0].imshow(Z.T, origin='lower', extent=ext, cmap='viridis', aspect='auto', norm=LogNorm())
ax[0].contour(alphas, betas, Z.T, levels=12, colors='white', linewidths=0.6, alpha=0.7)
ax[0].plot(0, 0, 'r*', ms=14, label=r'$\theta^\star$ (trained)')
for (name, (path, _)), c in zip(runs.items(), ['C1', 'C3', 'C2', 'w']):
    ax[0].plot(path[:, 0], path[:, 1], color=c, lw=1.4, label=name)
ax[0].set_xlabel(r'$\alpha$ (direction "from init")'); ax[0].set_ylabel(r'$\beta$ (random direction)')
ax[0].legend(fontsize=8); ax[0].set_title('the landscape, and the paths taken (log color)')
for (name, (_, ls)), c in zip(runs.items(), ['C1', 'C3', 'C2', 'k']):
    ax[1].semilogy(ls, color=c, lw=1.5, label=name)
ax[1].axhline(L_star, color='r', ls='--', lw=1.2, label=r'trained loss $L^*$')
ax[1].set_xlabel('step'); ax[1].set_ylabel('true loss along the path')
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
ax[1].set_title('the same race, loss vs step (goal: the red line)')
plt.tight_layout(); plt.show()

In [ ]:
# ---- animation: the full race, from the random start to the final loss value ----
fig, ax = plt.subplots(figsize=(6.8, 5), dpi=80)
ax.contourf(alphas, betas, Z.T, levels=20, cmap='viridis')
ax.contour(alphas, betas, Z.T, levels=10, colors='white', linewidths=0.5, alpha=0.6)
ax.plot(0, 0, 'r*', ms=16, label=r'trained $\theta^\star$ (loss $L^*$)')
ax.plot([s0], [0.0], 'ks', ms=6, label='random init')
trajs = {}
colors_ = {'SGD lr 1e-2': 'C1', 'SGD lr 5e-2': 'C3', 'SGD+momentum 5e-2': 'C2', 'Adam lr 2e-3': 'w'}
for name, c in colors_.items():
    (ln,) = ax.plot([], [], color=c, lw=1.6, label=name)
    (pt,) = ax.plot([], [], 'o', color=c, ms=5)
    trajs[name] = (ln, pt)
max_len = max(len(runs[n][1]) for n in runs)
stride = 7
ax.set_xlabel(r'$\alpha$'); ax.set_ylabel(r'$\beta$'); ax.legend(fontsize=8, loc='upper right')
title = ax.set_title('')

def update(f):
    step_ = min(f * stride, max_len - 1)
    for name, (ln, pt) in trajs.items():
        path, _ = runs[name]
        k = min(step_, len(path) - 1)
        ln.set_data(path[:k + 1, 0], path[:k + 1, 1])
        pt.set_data([path[k, 0]], [path[k, 1]])
    title.set_text(f'step {step_ + 1}/{max_len} — goal: the star, loss L* = {L_star:.3f}')

show_anim(fig, update, frames=max_len // stride + 1, interval=60)

**Why the trajectories look the way they do — the same reading as Lab 1.**

- **Adam (white) ARRIVES**: per-parameter step scaling is a diagonal preconditioner, and the
  FNO's complex spectral weights and real pointwise weights live at very different scales —
  exactly the geometry that rewards adaptive steps. This is why the FNO literature defaults
  to Adam(W), in either framework.
- **SGD + momentum (green) crawls** towards the goal; **SGD 5e-2 (red) moves but too slowly**;
  **SGD 1e-2 (orange) barely leaves the plateau** — at the random init the network outputs
  near-zero fields and the relative-$L^2$ surface is extremely flat for a small fixed step.

> **One sentence of theory** (again): plain GD converges only for
> $\eta < 2/\lambda_{\max}(H)$ on a locally quadratic loss; momentum straightens the effective
> step along valleys; Adam approximates diagonal preconditioning by $1/\sqrt{\text{EMA}(g^2)}$.
> **"Which optimizer works" is a property of the loss geometry, not of the framework.**

### §4.4 · Multiple minima — where you start decides which minimum you get

Same experiment as Lab 1 §4.4: train a second copy from a different seed (a *different* set of
weights with a comparable loss), plot the loss on the plane connecting the two trained
solutions, and let Adam fall from three random initializations.

In [ ]:
# ---- two trainings, two minima: the plane that connects them ----
fno_b, hist_b, _ = fit(FNO1dJAX(jax.random.PRNGKey(11)), epochs=60)
print(f'second training: validation rel-L2 {hist_b["val"][-1]:.4f} '
      f'(the first reached {hist["val"][-1]:.4f}) — different weights, comparable loss')

params_a, _ = eqx.partition(fno, eqx.is_array)
params_b, _ = eqx.partition(fno_b, eqx.is_array)
static_ab = static
d_ab = t_sub(params_a, params_b); s_ab = t_norm(d_ab); d_ab = t_scale(d_ab, 1.0 / s_ab)
leaves_ab, treedef_ab = jtu.tree_flatten(params_a)
kr2 = jax.random.split(jax.random.PRNGKey(2), len(leaves_ab))
d_perp = treedef_ab.unflatten([jax.random.normal(k, l.shape, l.dtype)
                               for k, l in zip(kr2, leaves_ab)])
dp = t_scale(d_perp, 1.0 / t_norm(d_perp))
dp = t_sub(dp, t_scale(d_ab, t_dot(dp, d_ab))); dp = t_scale(dp, 1.0 / t_norm(dp))
mid = jtu.tree_map(lambda x, y: (x + y) / 2, params_a, params_b)

@jax.jit
def plane_loss_mm(a, b):
    p = jtu.tree_map(lambda s_, u, v: s_ + a * u + b * v, mid, d_ab, dp)
    return rel_l2(batch_forward(eqx.combine(p, static_ab), xb), yb)

Gm = 17
als = np.linspace(-0.75 * s_ab, 0.75 * s_ab, Gm)
bes = np.linspace(-0.75 * s_ab, 0.75 * s_ab, Gm)
Zm = np.array([[float(plane_loss_mm(jnp.float32(a), jnp.float32(b))) for b in bes]
               for a in als])
print(f'two-minimum landscape: {Gm}x{Gm} evaluations; the models sit at +/- {s_ab/2:.1f} horizontally')

opt_mm = optax.adam(2e-3)

@jax.jit
def adam_step_mm(params, opt_state):
    loss, grads = jax.value_and_grad(
        lambda p: rel_l2(batch_forward(eqx.combine(p, static_ab), xb), yb))(params)
    updates, opt_state = opt_mm.update(grads, opt_state, params=params)
    return optax.apply_updates(params, updates), opt_state, loss

runs_mm = {}
for sd in (10, 11, 12):
    p0, _ = eqx.partition(FNO1dJAX(jax.random.PRNGKey(sd)), eqx.is_array)
    st_mm = opt_mm.init(p0)
    path, ls = [], []
    for _ in range(250):
        p0, st_mm, loss = adam_step_mm(p0, st_mm)
        path.append((float(t_dot(t_sub(p0, mid), d_ab)), float(t_dot(t_sub(p0, mid), dp))))
        ls.append(float(loss))
    runs_mm[f'random init {sd - 9}'] = (np.array(path, dtype=float), np.array(ls))
    print(f'random init {sd - 9}: loss {ls[0]:.3f} -> {ls[-1]:.4f}')

In [ ]:
# ---- animation: three initializations, three basins ----
fig, ax = plt.subplots(figsize=(7, 5.4), dpi=80)
ext_mm = [als[0], als[-1], bes[0], bes[-1]]
ax.imshow(Zm.T, origin='lower', extent=ext_mm, cmap='viridis', aspect='auto', norm=LogNorm())
ax.contour(als, bes, Zm.T, levels=10, colors='white', linewidths=0.5, alpha=0.6)
ax.plot([s_ab / 2], [0], 'r*', ms=17, label='trained model A')
ax.plot([-s_ab / 2], [0], '*', color='C1', ms=17, label='trained model B')
mm_trajs = {}
for (name, (path, _)), c in zip(runs_mm.items(), ['w', 'C2', 'C4']):
    (ln,) = ax.plot([], [], color=c, lw=1.7, label=name)
    (pt,) = ax.plot([], [], 'o', color=c, ms=5)
    mm_trajs[name] = (ln, pt)
ax.set_xlabel('direction connecting the two trained models')
ax.set_ylabel('random perpendicular direction')
ax.legend(fontsize=8, loc='upper right')
title = ax.set_title('')

def update(f):
    for name, (ln, pt) in mm_trajs.items():
        path, _ = runs_mm[name]
        k = min(f, len(path) - 1)
        ln.set_data(path[:k + 1, 0], path[:k + 1, 1])
        pt.set_data([path[k, 0]], [path[k, 1]])
    title.set_text(f'Adam step {f + 1}/250: each initialization falls into its own basin')

show_anim(fig, update, frames=250, interval=45)

**The reading, unchanged from Lab 1.** Two seeds, two minima, nearly the same
validation error: non-convexity is the *normal* geometry, (i) "the trained model" does not
exist — report over seeds; (ii) one run measures one basin, not the method.

### §4.5 · The same landscape in 3D

In [ ]:
# ---- the landscape and the trajectories in 3D ----
Zlog = np.log10(np.maximum(Z, 1e-4))
Agrid, Bgrid = np.meshgrid(alphas, betas)
fig = plt.figure(figsize=(9.5, 6.5))
ax3 = fig.add_subplot(projection='3d')
ax3.plot_surface(Agrid, Bgrid, Zlog.T, cmap='viridis', alpha=0.75,
                 linewidth=0, antialiased=True, rstride=1, cstride=1)
for (name, (path, ls)), c in zip(runs.items(), ['C1', 'C3', 'C2', 'w']):
    ax3.plot(path[:, 0], path[:, 1], np.log10(np.maximum(ls, 1e-4)), color=c, lw=1.8, label=name)
    ax3.plot([path[-1, 0]], [path[-1, 1]], [np.log10(max(ls[-1], 1e-4))], 'o', color=c, ms=5)
ax3.scatter([0], [0], [np.log10(max(L_star, 1e-4))], color='r', marker='*', s=160,
            label=r'trained $\theta^\star$')
ax3.set_xlabel(r'$\alpha$ (from init)'); ax3.set_ylabel(r'$\beta$ (random)')
ax3.set_zlabel(r'$\log_{10}$ loss'); ax3.view_init(elev=28, azim=-60)
ax3.legend(fontsize=8, loc='upper left')
ax3.set_title('the loss landscape with the optimizer trajectories, in 3D (JAX)')
plt.show()
print('Adam (white) dives to the red star; SGD 1e-2 (orange) stays on the high plateau.')

In [ ]:
# ---- animation: rotate the 3D landscape (the projection direction is a choice!) ----
fig = plt.figure(figsize=(7.2, 5.8), dpi=80)
ax3 = fig.add_subplot(projection='3d')
ax3.plot_surface(Agrid, Bgrid, Zlog.T, cmap='viridis', alpha=0.75, linewidth=0, antialiased=False)
for (name, (path, ls)), c in zip(runs.items(), ['C1', 'C3', 'C2', 'w']):
    ax3.plot(path[:, 0], path[:, 1], np.log10(np.maximum(ls, 1e-4)), color=c, lw=1.8)
ax3.scatter([0], [0], [np.log10(max(L_star, 1e-4))], color='r', marker='*', s=160)
ax3.set_xlabel(r'$\alpha$'); ax3.set_ylabel(r'$\beta$'); ax3.set_zlabel(r'$\log_{10}$ loss')

def update(f):
    ax3.view_init(elev=28, azim=-75 + f * 5)
    ax3.set_title(f'3D loss landscape, azimuth {-75 + f * 5} degrees')

show_anim(fig, update, frames=30, interval=120)

### §4.6 · The hyperparameter landscape (corner plot)

Same protocol as Lab 1 §4.6: 20 random configurations of (`modes`, `width`, `n_layers`,
`proj_hidden`), 25 epochs each — a coarse scan that exposes *interactions*, not optima.
On a MIG GPU the whole search takes well under a minute; on a laptop CPU, several minutes
(set `CORNER_RUNS` before opening the notebook to change the budget).

In [ ]:
# ---- random search over (modes, width, n_layers, proj_hidden) + corner plot ----
GRID = {'modes':       [4, 8, 16, 24, 32],
        'width':       [8, 16, 32, 48],           # also the width of the lifting map P
        'n_layers':    [2, 3, 4, 6],
        'proj_hidden': [16, 32, 64, 128]}          # the width of the projection head Q
rng = np.random.default_rng(7)
N_CORNER = int(os.environ.get('CORNER_RUNS', '20'))
seen, cfgs = set(), []
while len(cfgs) < N_CORNER:
    cfg = {k: int(rng.choice(v)) for k, v in GRID.items()}
    key_ = tuple(cfg.values())
    if key_ not in seen:
        seen.add(key_); cfgs.append(cfg)

print(f'{N_CORNER} configurations x 25 epochs:')
corner_loss = []
for i, cfg in enumerate(cfgs):
    m, h, _ = fit(FNO1dJAX(jax.random.PRNGKey(0), width=cfg['width'], modes=cfg['modes'],
                            n_layers=cfg['n_layers'], proj_hidden=cfg['proj_hidden']),
                  epochs=25, key_train=jax.random.PRNGKey(1))
    corner_loss.append(h['val'][-1])
    if (i + 1) % 5 == 0 or i == 0:
        print(f'  [{i + 1:2d}/{N_CORNER}] {cfg} -> val rel-L2 {h["val"][-1]:.4f}')
corner_loss = np.array(corner_loss)
best = int(corner_loss.argmin())
print('best configuration:', cfgs[best], '-> val rel-L2', round(float(corner_loss[best]), 4))

In [ ]:
# ---- the corner plot: 1-D marginals on the diagonal, 2-D pairs below, color = error ----
NAMES = list(GRID)
LBL = {'modes': 'modes m', 'width': 'width w (P width)', 'n_layers': 'layers L',
       'proj_hidden': 'proj_hidden (Q width)'}
fig, axes = plt.subplots(4, 4, figsize=(12, 10.5))
sc = None
for r in range(4):
    for c in range(4):
        ax = axes[r, c]
        xr = np.array([cfg[NAMES[r]] for cfg in cfgs])
        xc = np.array([cfg[NAMES[c]] for cfg in cfgs])
        if r == c:
            ax.scatter(xr, corner_loss, s=26, c='C0')
            ax.scatter(xr[best], corner_loss[best], marker='*', s=200, c='C3', zorder=5)
            ax.set_ylabel('val rel-L2'); ax.set_xlabel(LBL[NAMES[r]]); ax.grid(alpha=0.3)
        elif r > c:
            sc = ax.scatter(xc, xr, c=corner_loss, cmap='viridis', s=42, edgecolor='k', lw=0.3)
            ax.scatter(xc[best], xr[best], marker='*', s=220, c='C3', zorder=5,
                       edgecolor='k', lw=0.4)
            ax.set_xlabel(LBL[NAMES[c]]); ax.set_ylabel(LBL[NAMES[r]])
        else:
            ax.axis('off')
fig.colorbar(sc, ax=axes, shrink=0.55, label='validation rel-L2 (dark = better)')
fig.suptitle(f'Hyperparameter corner plot: {N_CORNER} architectures, 25 epochs each '
             '(red star = best of the search)', y=0.97)
plt.show()
print('Look for tilted clouds below the diagonal: modes without width to mix them are wasted.')

## §5 · Evaluate and super-resolve — the test of resolution invariance

Now — and only now — the untouched **test split** (H4), and the experiment no
grid-fixed network can do: evaluate the model trained at $N=64$ **directly at $N=256$**.
The spectral weights are indexed by mode, not by grid point — in JAX as in PyTorch.

In [ ]:
# ---- final evaluation on the untouched test split ----
Xte_j, Yte_j = enc(Xte_np), enc_y(Yte_np)
pred_test = np.array(dec_y(batch_forward(fno, Xte_j)))
rel = np.linalg.norm((pred_test - Yte_np).reshape(n_test, -1), axis=-1) / \
      np.linalg.norm(Yte_np.reshape(n_test, -1), axis=-1)
i_best, i_worst = int(rel.argmin()), int(rel.argmax())
fig, ax = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for a, i, t in zip(ax, [i_best, i_worst], ['best', 'worst']):
    a.plot(x_grid, Yte_np[i, 0], color='C0', lw=2, label='truth')
    a.plot(x_grid, pred_test[i, 0], color='C3', ls='--', lw=1.8, label='JAX FNO')
    a.set_title(f'{t} test sample (rel-L2 = {rel[i]:.3f})'); a.set_xlabel('x'); a.grid(alpha=0.3)
ax[0].legend(); ax[0].set_ylabel('u')
plt.show()
print(f'test rel-L2: mean {rel.mean():.4f} | median {np.median(rel):.4f} | worst {rel.max():.4f}')
print('(Lab 1 / PyTorch reported a mean of about 0.076 — JAX lands in the same range.)')

In [ ]:
# ---- zero-shot super-resolution: same model, finer grid, no retraining ----
N_HI = 256
Xhi_np, Yhi_np = DATA['X256'], DATA['Y256']        # unseen events at N=256, same seed as Lab 1
pred_hi = np.array(dec_y(batch_forward(fno, enc(Xhi_np))))
rel_hi = np.linalg.norm((pred_hi - Yhi_np).reshape(n_test, -1), axis=-1) / \
         np.linalg.norm(Yhi_np.reshape(n_test, -1), axis=-1)
x_hi = np.linspace(0, 1, N_HI, endpoint=False)

fig, ax = plt.subplots(1, 2, figsize=(11, 3), sharey=True)
ax[0].plot(x_grid, Yte_np[0, 0], color='C0', lw=2, label='truth')
ax[0].plot(x_grid, pred_test[0, 0], color='C3', ls='--', lw=1.6, label='JAX FNO')
ax[0].set_title(f'N=64 (training res) rel-L2={rel[0]:.4f}'); ax[0].legend(fontsize=8)
ax[1].plot(x_hi, Yhi_np[0, 0], color='C0', lw=2, label='truth (N=256)')
ax[1].plot(x_hi, pred_hi[0, 0], color='C3', ls='--', lw=1.6, label='JAX FNO (zero-shot)')
ax[1].set_title(f'N=256 (zero-shot) rel-L2={rel_hi[0]:.4f}'); ax[1].legend(fontsize=8)
for a in ax: a.set_xlabel('x'); a.grid(alpha=0.3)
plt.show()
print(f'resolution invariance, JAX edition: mean rel-L2 {rel.mean():.4f} at N=64, '
      f'{rel_hi.mean():.4f} at N=256 — the same model, no retraining')

## §6 · JAX vs PyTorch — the comparison

| | PyTorch (Labs 1–3) | JAX (this extension) |
|---|---|---|
| programming model | mutable modules, `backward()` | immutable pytrees, `jax.grad` |
| batching | DataLoader / index slicing | `jax.vmap` (batching is a *transform*) |
| speed | eager + `torch.compile` | `jax.jit` (XLA), first call compiles |
| optimizer | `torch.optim.AdamW` | `optax.adamw` (same algorithm, incl. masked weight decay) |
| modules | `nn.Module` | `equinox.Module` (a pytree with methods) |
| graph library | PyTorch Geometric | none needed here; JAX has no PyG equivalent |
| results here | val rel-L2 ~ 0.076 | the same, within seed noise |

**Takeaways.** (i) The *math* of operator learning is framework-independent — the spectral
convolution is four FFT lines in either. (ii) JAX's functional style makes the loss-landscape
machinery of §4 almost trivial: the model *is* a pytree, so interpolating, projecting and
re-combining parameters is pure `tree_map` — try rewriting §4 in PyTorch and you will feel
the difference. (iii) PyTorch's ecosystem (PyG!) and gentler learning curve remain the reason
the main labs of this school are in PyTorch. Choose the tool your problem's ecosystem needs.

## References

- Z. Li et al. *Fourier Neural Operator for Parametric PDEs.* ICLR 2021.
- N. Kovachki et al. *Neural Operator: Learning Maps Between Function Spaces with Applications to PDEs.* JMLR 24 (2023).
- H. Li et al. *Visualizing the Loss Landscape of Neural Nets.* NeurIPS 2018.


In [ ]:
if HEDY_SESSION is not None:
    HEDY_SESSION.report()
else:
    print('No Hedy session: nothing to report (laptop run).')